In [1]:
#노선 조회 API
#https://www.data.go.kr/iim/api/selectAPIAcountView.do
#오픈 API 정보
#https://www.data.go.kr/tcs/dss/selectApiDataDetailView.do?publicDataPk=15000193
#서울시교통정보과
#http://api.bus.go.kr/contents/sub02/getStaionByRoute.html
#서울교통공사 지하철혼잡도정보
#https://www.data.go.kr/data/15071311/fileData.do
#서울 지하철 호선별 역별 승하차 정보
#https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do

### IMPORT

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy
import networkx as nx
import requests
import folium
from geopy.geocoders import Nominatim
from urllib.parse import unquote
import apikey
from PIL import Image
import io

In [2]:
def getBusInfo(key,id): #버스정보 얻어오기
    url = 'http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute'
    bus_params = {'serviceKey':key,'busRouteId':id}
    response = requests.get(url,params=bus_params)
    return response

def getStationLoc(address): #주소 -> 위도, 경도 변환
    geolocator = Nominatim(user_agent="location translator")
    location = geolocator.geocode(address)
    if location:
        latitude = location.latitude
        longitude = location.longitude
        return latitude, longitude
    else:
        return None

In [5]:
r = getBusInfo(apikey.KEYS['decodingKey'],'100100124')
print(r.content)

http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?serviceKey=4bRs8jlCbYQ3QXTh7S928uX%2Bk7IQi2so9Q5PIXZlrXwNUWzq8eLhLx5cIxq5R9VGo7uVNwJr8E0MVRQQuZ8ASw%3D%3D&busRouteId=100100124


In [4]:
def naverMap(address): #네이버 지도 API활용해서 지도 출력
    client_id = apikey.NAVERKEYS['id']
    client_secret = apikey.NAVERKEYS['secret']

    url = 'https://naveropenapi.apigw.ntruss.com/map-static/v2/raster'
    headers = {
        "X-NCP-APIGW-API-KEY-ID": client_id,
        "X-NCP-APIGW-API-KEY": client_secret,
    }
    lon,lat = getStationLoc(address)
    params = {
        'w':500,
        'h':300,
        'center':f'{lon},{lat}',
        'level':16,
        'maptype':'satellite',
        'format':'png',
        'scale':1,
        'lang':'ko',
        'public_transit':True
    }

    res = requests.get(url, headers=headers, params=params)
    print(res.content)
    image_data = io.BytesIO(res.content)
    image = Image.open(image_data)
    image

In [5]:
naverMap('서울특별시 성동구 아차산로 100')

b'{"error":{"errorCode":"100","message":"Bad reqeust(coordinate transform fail)"}}'


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x161cb67a0>